# Tools

In [17]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pathlib import Path

from langchain_core.messages import HumanMessage, ToolMessage
from rich import print as rprint
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 从.env文件中加载环境变量
load_dotenv(Path('../.env'), override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model='qwen-max',
    model_provider='openai',
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
)


## 工具调用-ChatPromptTemplate封装提示词

In [22]:
from langchain_core.tools import tool


@tool(description="获取城市天气")
def get_weather(city: str):
    return f'{city}今天晴天，温度28度'


result = get_weather.invoke({'city': '北京'})
# print(result)

chat_model = model.bind_tools([get_weather])

config = {
    "run_name": "天气助手",  # 在LangSmith中这次运行会显示为
}

prompt = ChatPromptTemplate(
    [
        ('system', '你是一个智能助手'),
        MessagesPlaceholder(variable_name='history')
    ]
)
history = []
user_input = '深圳今天天气怎么样?'
history.append(('user', user_input))
prompt_value = prompt.invoke(
    {
        'history': history
    }
)

resp = chat_model.invoke(prompt_value, config=config)
history.append(resp)

# print(resp)
# print('-' * 40)

if resp.tool_calls:
    for call in resp.tool_calls:
        if call['name'] == 'get_weather':
            tool_result = get_weather.invoke(call['args'])
            tool_message = ToolMessage(
                content=tool_result,
                tool_call_id=call['id'],
                name=call['name']
            )
            history.append(tool_message)
    prompt_value = prompt.invoke({'history': history})
    final_resp = chat_model.invoke(prompt_value, config=config)
    history.append(final_resp)
    rprint(final_resp.content)
else:
    rprint(resp.content_blocks)



深圳今天是晴天，温度是28度。

## 工具调用-普通列表封装提示词

In [ ]:
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

# 声明一个消息列表
messages = [HumanMessage("今天北京天气如何")]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage到消息列表中
messages.append(response)

# rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 大模型和Agent的主要区别在于：大模型不会主动的调用工具，所以这时候我们需要主动让工具调用。
        # 返回的是ToolMessage类型消息，添加到消息列表中
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_7ac34297626e4e8882a79e)
 Call ID: call_7ac34297626e4e8882a79e
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京今天晴天，温度28度
=====================> messages <=====================
final_response: 
content='北京今天是晴天，温度是28度。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 265, 'total_tokens': 280, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-6c958461-c1a8-9c43-9896-8a